## 군집 분석 및 Plotly 시각화

이 노트북은 주어진 CSV 파일에 대해 군집 분석을 수행하고, 그 결과를 Plotly를 사용하여 2D 및 3D 산점도로 시각화합니다.

### 1. 데이터 로드

GitHub에서 CSV 파일을 직접 로드합니다.

In [ ]:
import pandas as pd
import numpy as np
import ipywidgets as widgets

from IPython.display import display, HTML, clear_output

import plotly.express as px
import plotly.graph_objects as go

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans



# @title
github_csv_url = "https://github.com/CarlosQuperman/AIEDAP2026_LOCAL_DATA/raw/refs/heads/main/5%EC%9E%A5/2025_%EA%B0%95%EC%9B%90%ED%8A%B9%EB%B3%84%EC%9E%90%EC%B9%98%EB%8F%84_%EC%8B%9C%EA%B5%B0%EA%B5%AC%EB%B3%84%20%EC%9D%B8%EA%B5%AC.csv"
df = pd.read_csv(github_csv_url,encoding='euc-kr')

# 데이터 확인
display(df.head())
display(df.info())

### 2. 데이터 전처리 및 군집 분석 (대화형)

군집 분석을 위해 사용자가 직접 숫자형 데이터를 선택하고, K-Means 군집의 개수를 지정할 수 있도록 `ipywidgets`를 활용합니다. 체크 박스를 통해 군집 분석을 위한 변수(속성)과 군집의 개수를 설정하고 시각화를 위한 X,Y,Z를 설정한 후 녹색 버튼을 누르세요

In [ ]:
# @title
import pandas as pd
import numpy as np
import ipywidgets as widgets

from IPython.display import display, HTML, clear_output

import plotly.express as px
import plotly.graph_objects as go

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


# ----------------------------------
# 저장된 HTML 파일을 불러와 표시하는 함수
# 필요할 때 별도 셀에서 사용 가능
# ----------------------------------
def display_saved_html(filename):
    with open(filename, "r", encoding="utf-8") as f:
        html_content = f.read()
    display(HTML(html_content))


# ----------------------------------
# 군집 영역을 원으로 표시하는 함수
# 2D 시각화에만 사용
# ----------------------------------
def add_cluster_circles(fig, plot_df, x_col, y_col, cluster_col="Cluster", padding=1.15):
    for cluster in sorted(plot_df[cluster_col].unique()):
        cluster_df = plot_df[plot_df[cluster_col] == cluster]

        if len(cluster_df) < 2:
            continue

        x_center = cluster_df[x_col].mean()
        y_center = cluster_df[y_col].mean()

        distances = np.sqrt(
            (cluster_df[x_col] - x_center) ** 2 +
            (cluster_df[y_col] - y_center) ** 2
        )

        radius = distances.max() * padding

        if radius == 0 or np.isnan(radius):
            continue

        theta = np.linspace(0, 2 * np.pi, 160)
        circle_x = x_center + radius * np.cos(theta)
        circle_y = y_center + radius * np.sin(theta)

        fig.add_trace(
            go.Scatter(
                x=circle_x,
                y=circle_y,
                mode="lines",
                fill="toself",
                opacity=0.18,
                line=dict(width=3),
                name=f"군집 {cluster} 원형 영역",
                showlegend=True
            )
        )

    return fig


# ----------------------------------
# 기본 설정
# ----------------------------------
available_numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()

name_col = "행정구역"
size_col = "2025년_총인구수"
result_col = "군집결과 그룹 이름"

csv_path = "/content/cluster_result.csv"
path_2d = "/content/plot_user_axis_2d.html"
path_3d = "/content/plot_user_axis_3d.html"

if name_col not in df.columns:
    print(f"주의: '{name_col}' 컬럼이 없습니다.")

if size_col not in df.columns:
    print(f"주의: '{size_col}' 컬럼이 없습니다. 점 크기 기능은 비활성화됩니다.")


# ----------------------------------
# 군집 분석 속성 체크박스
# ----------------------------------
feature_checkboxes = [
    widgets.Checkbox(
        value=True,
        description=col,
        indent=False
    )
    for col in available_numeric_cols
]

checkbox_box = widgets.VBox(
    feature_checkboxes,
    layout=widgets.Layout(
        width="450px",
        max_height="260px",
        overflow="auto",
        border="1px solid lightgray",
        padding="8px"
    )
)


# ----------------------------------
# K 선택
# ----------------------------------
k_slider = widgets.IntSlider(
    value=4,
    min=2,
    max=9,
    step=1,
    description="군집 수 K:",
    continuous_update=False
)


# ----------------------------------
# 시각화 축 선택
# ----------------------------------
x_axis_dropdown = widgets.Dropdown(
    options=available_numeric_cols,
    value=available_numeric_cols[0],
    description="X축:"
)

y_axis_dropdown = widgets.Dropdown(
    options=available_numeric_cols,
    value=available_numeric_cols[1] if len(available_numeric_cols) > 1 else available_numeric_cols[0],
    description="Y축:"
)

z_axis_dropdown = widgets.Dropdown(
    options=available_numeric_cols,
    value=available_numeric_cols[2] if len(available_numeric_cols) > 2 else available_numeric_cols[0],
    description="Z축:"
)


run_button = widgets.Button(
    description="군집 분석 수행 및 저장",
    button_style="success"
)

output_widget = widgets.Output()


# ----------------------------------
# 선택된 군집 분석 속성 가져오기
# ----------------------------------
def get_selected_features():
    selected = []

    for cb in feature_checkboxes:
        if cb.value:
            selected.append(cb.description)

    return list(dict.fromkeys(selected))


# ----------------------------------
# 메인 실행 함수
# ----------------------------------
def run_clustering(button):
    with output_widget:
        clear_output(wait=True)

        selected_features = get_selected_features()

        x_col = x_axis_dropdown.value
        y_col = y_axis_dropdown.value
        z_col = z_axis_dropdown.value

        k_value = k_slider.value

        if len(selected_features) < 2:
            print("군집 분석에는 최소 2개 이상의 숫자형 속성을 선택해주세요.")
            return

        if x_col == y_col:
            print("X축과 Y축은 서로 다른 속성으로 선택해주세요.")
            return

        # X, Y축은 시각화에 필요하므로 분석 속성에도 자동 포함
        for col in [x_col, y_col]:
            if col not in selected_features:
                selected_features.append(col)

        # 분석 속성이 3개 이상일 때만 Z축을 3D 시각화에 활용
        use_3d_file = False

        if len(selected_features) >= 3:
            if z_col not in selected_features:
                selected_features.append(z_col)

            if len(set([x_col, y_col, z_col])) == 3:
                use_3d_file = True

        # 필요한 컬럼 구성
        required_cols = selected_features.copy()

        if size_col in df.columns and size_col not in required_cols:
            required_cols.append(size_col)

        if name_col in df.columns and name_col not in required_cols:
            required_cols.append(name_col)

        required_cols = list(dict.fromkeys(required_cols))

        data = df[required_cols].copy()

        # 군집 분석에 필요한 숫자형 속성 결측값만 제거
        # 이때 data.index는 원본 df의 인덱스를 유지함
        data = data.dropna(subset=selected_features)

        if data.empty:
            print("선택된 속성 조합으로 분석 가능한 데이터가 없습니다.")
            return

        if len(data) < k_value:
            print("데이터 개수보다 군집 수 K가 큽니다. K 값을 줄여주세요.")
            return

        # ----------------------------------
        # K-Means 군집 분석
        # ----------------------------------
        clustering_data = data[selected_features].copy()

        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(clustering_data)

        kmeans = KMeans(
            n_clusters=k_value,
            random_state=42,
            n_init=10
        )

        clusters = kmeans.fit_predict(scaled_data)

        # ----------------------------------
        # 원본 df에 군집 결과 반영 후 CSV 저장
        # ----------------------------------
        result_df = df.copy()
        result_df[result_col] = ""

        result_df.loc[data.index, result_col] = [
            f"군집 {cluster}" for cluster in clusters
        ]

        result_df.to_csv(
            csv_path,
            index=False,
            encoding="utf-8-sig"
        )

        globals()["cluster_result_df"] = result_df

        # ----------------------------------
        # 시각화용 데이터프레임
        # ----------------------------------
        plot_df = data.copy()
        plot_df["Cluster"] = clusters.astype(str)

        hover_col = name_col if name_col in plot_df.columns else None
        size_argument = size_col if size_col in plot_df.columns else None

        hover_data_cols = []

        if name_col in plot_df.columns:
            hover_data_cols.append(name_col)

        if size_col in plot_df.columns:
            hover_data_cols.append(size_col)

        for col in selected_features:
            if col in plot_df.columns and col not in hover_data_cols:
                hover_data_cols.append(col)

        # ----------------------------------
        # 2D 시각화 생성 및 HTML 저장
        # ----------------------------------
        fig_2d = px.scatter(
            plot_df,
            x=x_col,
            y=y_col,
            color="Cluster",
            size=size_argument,
            hover_name=hover_col,
            hover_data=hover_data_cols,
            title=f"사용자 지정 축 2D 군집 시각화 (K={k_value})<br>X축: {x_col}, Y축: {y_col}, 크기: {size_col}",
            labels={
                x_col: x_col,
                y_col: y_col,
                "Cluster": "군집",
                size_col: "2025년 총인구수",
                name_col: "행정구역"
            },
            color_discrete_sequence=px.colors.qualitative.Pastel,
            size_max=50
        )

        fig_2d.update_traces(
            marker=dict(
                opacity=0.75,
                line=dict(width=1.2, color="DarkSlateGrey")
            )
        )

        fig_2d = add_cluster_circles(
            fig_2d,
            plot_df,
            x_col,
            y_col,
            cluster_col="Cluster",
            padding=1.18
        )

        fig_2d.update_layout(
            width=1100,
            height=720,
            legend_title_text="군집",
            template="plotly_white"
        )

        fig_2d.write_html(path_2d, include_plotlyjs="cdn")

        # ----------------------------------
        # 3D 시각화 생성 및 HTML 저장
        # 단, 분석 속성이 3개 미만이거나 축이 중복되면 생성하지 않음
        # ----------------------------------
        if use_3d_file:
            fig_3d = px.scatter_3d(
                plot_df,
                x=x_col,
                y=y_col,
                z=z_col,
                color="Cluster",
                size=size_argument,
                hover_name=hover_col,
                hover_data=hover_data_cols,
                title=f"사용자 지정 축 3D 군집 시각화 (K={k_value})<br>X축: {x_col}, Y축: {y_col}, Z축: {z_col}, 크기: {size_col}",
                labels={
                    x_col: x_col,
                    y_col: y_col,
                    z_col: z_col,
                    "Cluster": "군집",
                    size_col: "2025년 총인구수",
                    name_col: "행정구역"
                },
                color_discrete_sequence=px.colors.qualitative.Pastel,
                size_max=35
            )

            fig_3d.update_traces(
                marker=dict(
                    opacity=0.75,
                    line=dict(width=1)
                )
            )

            fig_3d.update_layout(
                width=1100,
                height=720,
                legend_title_text="군집",
                template="plotly_white"
            )

            fig_3d.write_html(path_3d, include_plotlyjs="cdn")

        # ----------------------------------
        # 결과 메시지
        # ----------------------------------
        print("군집 분석, CSV 저장, HTML 시각화 파일 저장이 완료되었습니다.")

        print(f"\nCSV 저장 파일:")
        print(csv_path)

        print(f"\n2D 시각화 HTML 저장 파일:")
        print(path_2d)

        if use_3d_file:
            print(f"\n3D 시각화 HTML 저장 파일:")
            print(path_3d)
        else:
            print("\n3D 시각화 HTML은 생성하지 않았습니다.")
            print("이유: 군집 분석 속성이 3개 미만이거나 X/Y/Z축이 서로 다르지 않습니다.")
            print("Z축 선택값은 무시되었습니다.")

        print("\nColab에서 파일 확인 방법:")
        print("1. 왼쪽 폴더 아이콘 클릭")
        print("2. /content 폴더 열기")
        print("3. cluster_result.csv, plot_user_axis_2d.html, plot_user_axis_3d.html 확인")

        print("\n노트북에서 결과 데이터프레임 확인:")
        print("cluster_result_df.head()")



# ----------------------------------
# 버튼 연결
# ----------------------------------
run_button.on_click(run_clustering)


# ----------------------------------
# UI 출력
# ----------------------------------
ui = widgets.VBox([
    widgets.HTML("<h3>군집 분석 설정</h3>"),

    widgets.HTML("<b>1. 군집 분석에 사용할 속성 선택</b>"),
    checkbox_box,

    widgets.HTML("<br><b>2. 군집 수 선택</b>"),
    k_slider,

    widgets.HTML("<br><b>3. 시각화 축 직접 선택</b>"),
    widgets.HTML("※ 2D/3D 시각화는 HTML 파일로 저장됩니다. 화면 표시는 별도 셀에서 실행하세요."),
    widgets.HBox([x_axis_dropdown, y_axis_dropdown, z_axis_dropdown]),

    widgets.HTML("<br>"),
    run_button,

    output_widget
])

display(ui)

In [ ]:
# @title
from IPython.display import HTML, display

# 2D
with open("/content/plot_user_axis_2d.html", "r", encoding="utf-8") as f:
    display(HTML(f.read()))


In [ ]:
# @title
with open("/content/plot_user_axis_3d.html", "r", encoding="utf-8") as f:
    display(HTML(f.read()))